# 01 · zerozero.pt data pipeline

Recover the fixtures of a **specific season** from zerozero.pt and turn them into the
tidy tables every other notebook reads.

```
zerozero.pt  ──scrape_season──▶  data/seasons/<key>/raw/       one file per team
                                   <slug>_fixtures.csv            league rows only
                                   <slug>_all_competitions.csv    everything (local only)
             ──process_season─▶  data/seasons/<key>/processed/
                                   fixtures_played.csv            deduplicated
                                   fixtures_remaining.csv
             ──validate_season▶  coverage and consistency checks
```

**Scraping is off by default** (`RUN_SCRAPE = False`), so the notebook runs end to end on the
data already in the repo. Turn it on to fetch fresh data; it needs Chrome and the `selenium`
package. Re-running is safe: teams already on disk are skipped unless `OVERWRITE = True`, so a
run cut short by zerozero's rate limit just continues where it stopped.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))  # works without `pip install -e .`

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

from season_model.config import SEASONS, define_season, epoca_id_for, get_season
from season_model.pipeline import (
    competitions_found, load_season, process_season, scrape_season, validate_season,
)

## 1. Choose the season

A season is a `SeasonConfig`: the league's exact zerozero competition name, one fixtures-page
URL per team, and the zerozero season id **`epoca_id`**. Every URL is pinned to that id before
scraping. A "jogos" URL without it shows the *current* season, so re-scraping 2025/26 during
2026/27 would otherwise fetch the wrong year without any error.

In [ ]:
SEASON_KEY = "mansores_2025_26"   # any key in SEASONS, or one you define below
RUN_SCRAPE = False                # True = fetch from zerozero.pt (needs Chrome + selenium)
OVERWRITE = False                 # True = re-scrape teams already on disk
TEAMS = None                      # None = every team, or e.g. ["mansores", "crc_vale"]

print("Registered seasons:", ", ".join(SEASONS))
season = get_season(SEASON_KEY)
print(f"\n{season.label}\ncompetition: {season.competition_name}\nepoca_id:    {season.epoca_id}")
pd.DataFrame({"team": list(season.team_urls), "url (pinned)": list(season.team_urls.values())})

### Adding another season without editing `config.py`

`define_season()` registers a season for this session. `epoca_id_for(start_year)` extrapolates
the zerozero id from the two known ones (154 = 2024/25, 155 = 2025/26); check it against the
"Época" selector on any team page before trusting it further out. The competition name is a
placeholder until the first scrape: section 2 lists every competition found, then set the real
one. Once it works, copy the config into `SEASONS` in `src/season_model/config.py`.

In [ ]:
# Example only: the 2026/27 division of the same club. Team list and competition name must be
# checked on zerozero (promotions/relegations change the league).
example = define_season(
    key="mansores_2026_27",
    label="Mansores — 2026/27 (example, unverified)",
    competition_name="TODO: copy from competitions_found()",
    primary_team="mansores",
    teams={"mansores": "https://www.zerozero.pt/equipa/mansores/10972/jogos?grp=1"},
    epoca_id=epoca_id_for(2026),
    register=False,   # True to make it selectable via SEASON_KEY
)
example.team_urls

## 2. Scrape

One headless Chrome session, one page per team, a pause between teams. It stops at the first
rate-limit page ("Serviço Temporariamente Suspenso") instead of retrying in a loop: wait a few
minutes and run the cell again.

In [ ]:
if RUN_SCRAPE:
    report = scrape_season(season, teams=TEAMS, overwrite=OVERWRITE)
    display(report.to_frame())
    if report.rate_limited:
        print("Rate limited: wait a few minutes, then re-run this cell (finished teams are skipped).")
else:
    on_disk = sorted(p.name for p in season.raw_dir.glob("*.csv"))
    print(f"RUN_SCRAPE is False, using what is on disk in {season.raw_dir.relative_to(ROOT)}:")
    print("\n".join(f"  {name}" for name in on_disk) or "  (nothing)")

In [ ]:
# Which competitions did the scrape see? Use this to set competition_name for a new season.
comps = competitions_found(season)
comps if not comps.empty else print("No *_all_competitions.csv files yet (they are written by scrape_season).")

## 3. Process

Each team's page lists every match from its own point of view, so a match between two scraped
teams appears twice. `process_season` keeps the league rows only (the Mansores export also holds
two Taça Pecol ties), turns `Location` into explicit home/away teams, reads `Result` as
**home-away** (not "our team first", the bug fixed in the original port) and deduplicates.

`fermedo_2024_25` predates the per-team raw layout: its raw folder holds a single already-tidy
export, so it is read from `processed/` directly.

In [ ]:
try:
    played, remaining = process_season(season)
    print(f"Processed from raw -> {season.processed_dir.relative_to(ROOT)}")
except FileNotFoundError as exc:
    print(f"{exc}\nFalling back to the stored processed tables.")
    played, remaining = load_season(season)

print(f"{len(played)} played, {len(remaining)} remaining")
played.head()

## 4. Validate

The check that matters most for modelling is **coverage**. A season scraped from one team's page
only knows that team's matches; every other team's strength is then estimated from one game
against it, and the prediction for the rest of the table is close to noise.

In [ ]:
checks = validate_season(season, played, remaining)
checks

In [ ]:
# Every registered season at a glance.
rows = []
for key, s in SEASONS.items():
    p, r = load_season(s)
    v = validate_season(s, p, r).set_index("check")
    rows.append({
        "season": key,
        "epoca_id": s.epoca_id,
        "teams": s.n_teams,
        "played": len(p),
        "remaining": len(r),
        "coverage": v.loc["coverage: matches vs full double round-robin", "value"],
        "complete": bool(v.loc["coverage: matches vs full double round-robin", "ok"]) and r.empty,
    })
pd.DataFrame(rows)

## 5. Next steps for the data

* **`mansores_2025_26` is partial and stale**: Mansores' own fixtures only, scraped up to
  2026-01-17. Re-scrape all 12 teams with `RUN_SCRAPE = True, OVERWRITE = True`. Because the URLs
  are pinned to `epoca_id = 155`, this recovers the full, finished 2025/26 season even now, and
  gives the evaluation notebook a second complete season to backtest on.
* **Player-level data** (lineups, goals, cards, estimated minutes) is available through
  `season_model.scraping.zerozero_client.scrape_season_match_reports(match_urls)` and
  `season_model.players.match_events`. It is not part of the season model.